## **07_mlp: The "Thinking" Layer: The MLP**

We have built `CausalSelfAttention` — the **communication** layer where tokens exchange information.  
But gathering information is only half the battle.  
After each token has collected context, it needs time to **think** about it.

The **MLP** (Multi-Layer Perceptron) processes each token's information **independently**.  

**Analogy:** The attention layer was a group meeting where everyone shared ideas.  
The MLP is like each person going back to their desk to sit and think about what they just heard.

### The Architecture: Expand and Contract

1. **Expansion** (`fc`): Project from `n_embd` up to `4 * n_embd`
2. **Non-linearity** (`gelu`): GPT-2 uses GELU, a smooth alternative to ReLU
3. **Contraction** (`proj`): Project back down to `n_embd`
4. **Dropout** (`drop`): Regularization to prevent overfitting

```python
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.fc   = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.drop = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.fc(x)
        x = F.gelu(x)
        x = self.drop(self.proj(x))
        return x
```

That's it! After the complexity of attention, this is refreshingly simple.

### Part 1: Demystifying `nn.Linear`

Before building the full MLP, let's demystify its core component.  
At its heart, `nn.Linear` is just: **`output = input @ W^T + b`**  

No magic at all. Let's prove it.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

C_in = 2
C_out = 4
linear_layer = nn.Linear(C_in, C_out)

# Learnable parameters:
# Weights: shape (C_out, C_in) = (4, 2) -> 8 params
# Biases:  shape (C_out) = (4)         -> 4 params

# Set them manually to see the math clearly
linear_layer.weight.data = torch.tensor([
    [1., 0.],   # Weights for output element 0
    [-1., 0.],  # Weights for output element 1
    [0., 2.],   # Weights for output element 2
    [0., -2.]   # Weights for output element 3
])
linear_layer.bias.data = torch.tensor([1., 1., -1., -1.])

print("Weights shape:", linear_layer.weight.shape)
print("Bias shape:", linear_layer.bias.shape)

Let's manually calculate the first output element:

`output[0] = (input[0] * weight[0,0]) + (input[1] * weight[0,1]) + bias[0]`  
`output[0] = (0.5 * 1.0) + (-0.5 * 0.0) + 1.0 = 1.5`

In [ ]:
input_vector = torch.tensor([0.5, -0.5])
output_vector = linear_layer(input_vector)

print("Input vector:", input_vector)
print("Output vector:", output_vector)

The first element is `1.5` — matches our manual calculation. No magic.

### Part 2: Full MLP Walkthrough with Numbers

Let's trace a single token's vector through the entire MLP.

+ Embedding dimension `C = 2`
+ Intermediate dimension: `4 * C = 8`
+ Input: a single token vector `[0.5, -0.5]`

In [ ]:
# Our input vector for one token. Shape (B, T, C) -> (1, 1, 2)
x = torch.tensor([[[0.5, -0.5]]])
print("Input:", x, "Shape:", x.shape)

#### Step 1: The Expansion Layer (`fc`)

Project from `C=2` up to `4*C=8`.

In [ ]:
torch.manual_seed(42)
fc = nn.Linear(2, 8)
fc.weight.data = torch.randn(8, 2) * 2
fc.bias.data = torch.ones(8)

x_expanded = fc(x)
print("--- After Expansion Layer ---")
print("Shape:", x_expanded.shape)
print("Values:\n", x_expanded.data.round(decimals=2))

Our 2-dimensional vector has been expanded to 8 dimensions.

#### Step 2: The GELU Activation

GELU is a smoother version of ReLU.  
It squashes negative values towards zero but allows a small amount of negative signal through.  
Positive values are largely unchanged.

| Input | GELU(Input) |
| :--- | :--- |
| 2.4 | ~2.39 |
| 1.0 | ~0.84 |
| 0.0 | 0.0 |
| -0.5 | ~-0.15 |
| -2.0 | ~-0.00 |

In [ ]:
x_activated = F.gelu(x_expanded)
print("--- After GELU Activation ---")
print("Shape:", x_activated.shape)
print("Values:\n", x_activated.data.round(decimals=2))

Large positive values are almost untouched, large negative values squashed to nearly zero.  
This non-linear step is essential for learning complex patterns.

#### Step 3: The Contraction Layer (`proj`)

Project the 8-dimensional vector back down to our original `C=2`.

In [ ]:
torch.manual_seed(123)
proj = nn.Linear(8, 2)
proj.weight.data = torch.randn(2, 8)
proj.bias.data = torch.zeros(2)

x_projected = proj(x_activated)
print("--- After Contraction Layer ---")
print("Shape:", x_projected.shape)
print("Values:\n", x_projected.data.round(decimals=2))

We are back to our original shape `(1, 1, 2)`.

#### Step 4: Dropout

**During training:** randomly sets 10% of elements to zero (regularization).  
**During inference** (`model.eval()`): does nothing, passes data through unchanged.

In [ ]:
drop = nn.Dropout(0.1)
final_output = drop(x_projected)
print("--- After Dropout ---")
print("Shape:", final_output.shape)
print("Values:\n", final_output.data.round(decimals=2))

### Key Takeaway

The MLP transforms the input vector while **preserving its shape** `(B, T, C)`.  
This is critical because it allows us to:
1. **Add** this output back to the original input (a "residual connection")
2. **Stack** multiple Transformer blocks on top of each other

We now have both major components of our Transformer block:
+ `CausalSelfAttention` — **communication** (tokens share information)
+ `MLP` — **thinking** (each token processes independently)

Next: how to assemble them using **residual connections**.